In [ ]:
# evaluation/analysis.ipynb
# Copy each block below as a separate notebook cell

# ── Cell 1: Imports & style ───────────────────────────────────────────────────
import os
import sys
sys.path.insert(0, os.path.abspath(".."))
from visuals.plot_env import (
    plot_episode,
    plot_phase_reward_structure,
    plot_height_curves,
)

from visuals.plot_utils      import set_style
from training.train          import load_config
from evaluation.eval         import (
    load_all_logs,
    compute_summary,
    compute_learning_curves,
    compute_arch_usage,
    compute_switch_frequency,
    compute_height_curves,
)
from visuals.plot_curves     import (
    plot_learning_curves,
    plot_per_seed_curves,
    plot_reward_distributions,
    plot_reward_vs_height,
)
from visuals.plot_switching  import (
    plot_arch_usage,
    plot_switch_frequency,
    plot_episode_timeline,
    plot_switching_per_phase,
)
from visuals.plot_comparison import (
    plot_final_comparison,
    plot_height_comparison,
    plot_switches_vs_reward,
    plot_summary_grid,
)
from visuals.plot_env        import (
    plot_episode,
    plot_phase_reward_structure,
    plot_height_curves,
)
from visuals.plot_robot      import (
    plot_cube_trajectory,
    plot_action_analysis,
    plot_arch_preference_per_phase,
    plot_lift_progress,
)
from core.robosuite_env      import RobosuitePhased

set_style()
cfg = load_config()
print("Config loaded.")

In [ ]:
# ── Cell 2: Load all logs ─────────────────────────────────────────────────────
df           = load_all_logs(cfg)
summary      = compute_summary(df)
curves       = compute_learning_curves(df)
arch_usage   = compute_arch_usage(df)
switch_freq  = compute_switch_frequency(df)
height_curves = compute_height_curves(df)

print(f"Agents:  {df['agent'].unique().tolist()}")
print(f"Seeds:   {sorted(df['seed'].unique().tolist())}")
print(f"Rows:    {len(df)}")
print()
print(summary.to_string(index=False))

In [ ]:
# ── Cell 3: Phase reward structure ───────────────────────────────────────────
plot_phase_reward_structure(
    save_path="../results/phase_reward_structure.png",
    show=True,
)


In [ ]:
# ── Cell 4: Sample episode — random policy ───────────────────────────────────
env    = RobosuitePhased(seed=42)
obs, _ = env.reset(seed=42)
done   = False
while not done:
    obs, reward, terminated, truncated, _ = env.step(env.action_space.sample())
    done = terminated or truncated

history    = env.get_history_arrays()
boundaries = env.get_phase_boundaries()
env.close()

plot_episode(
    history, boundaries,
    title="Robosuite Episode — Random Policy (seed=42)",
    save_path="../results/env_episode.png",
    show=True,
)


In [ ]:
# ── Cell 5: Cube trajectory analysis ────────────────────────────────────────
plot_cube_trajectory(
    history, boundaries,
    save_path="../results/cube_trajectory.png",
    show=True,
)

In [ ]:
# ── Cell 6: Action analysis ──────────────────────────────────────────────────
plot_action_analysis(
    history, boundaries,
    save_path="../results/action_analysis.png",
    show=True,
)

In [ ]:
# ── Cell 7: Learning curves — all agents ────────────────────────────────────
plot_learning_curves(
    curves,
    save_path="../results/learning_curves.png",
    show=True,
)

In [ ]:
# ── Cell 8: Reward vs cube height ────────────────────────────────────────────
plot_reward_vs_height(
    df, agent="self_modifying",
    save_path="../results/reward_vs_height.png",
    show=True,
)

In [ ]:
# ── Cell 9: Per-seed curves ──────────────────────────────────────────────────
plot_per_seed_curves(
    df, agent="self_modifying",
    save_path="../results/per_seed_curves.png",
    show=True,
)

In [ ]:
# ── Cell 10: Reward distributions ────────────────────────────────────────────
plot_reward_distributions(
    df,
    save_path="../results/reward_distributions.png",
    show=True,
)

In [ ]:
# ── Cell 11: Architecture usage over training ────────────────────────────────
plot_arch_usage(
    arch_usage,
    save_path="../results/arch_usage.png",
    show=True,
)

In [ ]:
# ── Cell 12: Switch frequency ────────────────────────────────────────────────
plot_switch_frequency(
    switch_freq,
    save_path="../results/switch_frequency.png",
    show=True,
)

In [ ]:
# ── Cell 13: Switching per phase ─────────────────────────────────────────────
plot_switching_per_phase(
    df,
    save_path="../results/switching_per_phase.png",
    show=True,
)

In [ ]:
# ── Cell 14: Architecture preference per phase ───────────────────────────────
plot_arch_preference_per_phase(
    df,
    save_path="../results/arch_preference_per_phase.png",
    show=True,
)

In [ ]:
# ── Cell 15: Lift progress — all agents ──────────────────────────────────────
plot_lift_progress(
    height_curves,
    save_path="../results/lift_progress.png",
    show=True,
)

In [ ]:
# ── Cell 16: Final performance comparison ────────────────────────────────────
plot_final_comparison(
    summary,
    save_path="../results/final_comparison.png",
    show=True,
)

In [ ]:
# ── Cell 17: Cube height comparison ──────────────────────────────────────────
plot_height_comparison(
    summary,
    save_path="../results/height_comparison.png",
    show=True,
)

In [ ]:
# ── Cell 18: Switches vs reward ──────────────────────────────────────────────
plot_switches_vs_reward(
    df,
    save_path="../results/switches_vs_reward.png",
    show=True,
)

In [ ]:
# ── Cell 19: Summary grid ────────────────────────────────────────────────────
plot_summary_grid(
    summary,
    save_path="../results/summary_grid.png",
    show=True,
)


In [ ]:
# ── Cell 20: Episode timeline — self-modifying ───────────────────────────────
from core.policy          import load_policy
from core.meta_controller import MetaController
from training.train       import run_self_modifying_episode

env      = RobosuitePhased(seed=0)
pol_cfg  = cfg["inner_policies"]
tr_cfg   = cfg["training"]
meta_cfg = cfg["meta_controller"]

policies = {
    "mlp":  load_policy("mlp",  os.path.join(tr_cfg["models_dir"], "mlp_pretrained"),  env, pol_cfg["device"]),
    "lstm": load_policy("lstm", os.path.join(tr_cfg["models_dir"], "lstm_pretrained"), env, pol_cfg["device"]),
}

meta = MetaController(**{k: meta_cfg[k] for k in [
    "n_archs", "meta_state_dim", "hidden", "lr",
    "gamma", "switching_cost", "decision_interval",
    "history_window", "device",
]})
meta.load(os.path.join(tr_cfg["models_dir"], "meta_seed0_final.pt"))

ep_reward, arch_log, switches, cube_heights = run_self_modifying_episode(
    env, policies, meta, cfg, deterministic=True
)
history    = env.get_history_arrays()
boundaries = env.get_phase_boundaries()

print(f"Episode reward: {ep_reward:.3f} | Switches: {switches}")

plot_episode_timeline(
    history, arch_log,
    save_path="../results/episode_timeline.png",
    show=True,
)
env.close()

In [ ]:
# ── Cell 21: Save summary ────────────────────────────────────────────────────
out = os.path.join(tr_cfg["logs_dir"], "summary.csv")
summary.to_csv(out, index=False)
print(f"Summary saved → {out}")
print(f"All figures saved to results/")
print("Done.")